Setup Validation

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

1. Imports & versions

In [ ]:
import torch, mlflow, sklearn, numpy, pandas, psutil
from src.utils.system_info import get_system_info
print('torch  ', torch.__version__, '| cuda', torch.cuda.is_available())
print('mlflow ', mlflow.__version__)
print('sklearn', sklearn.__version__)
get_system_info()

2. Configs load

In [ ]:
from src.utils.config import load_configs
configs = load_configs()
print('Sections:', list(configs))
print('Algorithm:', configs['experiment']['algorithm'])
print('Seeds    :', len(configs['experiment']['seeds']))
print('Budget   :', configs['experiment']['budget'])

3. Raw data present

In [ ]:
from src.data import loader
from src.utils import paths
for which in ('train', 'test'):
    p = paths.resolve(configs['dataset']['dataset']['root_dir']) / configs['dataset']['dataset'][f'{which}_file']
    print(f'{which:5s}', 'EXISTS' if p.exists() else 'MISSING', p)
labels = loader.load_labels(configs['dataset'], 'train')
import numpy as np
print('train label counts:', dict(zip(*np.unique(labels, return_counts=True))))

4. Fixed splits

In [ ]:
from src.data import split_loader
train_idx, val_idx, test_idx = split_loader.load_splits()
print('train/val/test sizes:', len(train_idx), len(val_idx), len(test_idx))

In [ ]:
from src.tracking import database, artifact_manager, mlflow_logger
paths.ensure_dirs()
artifact_manager.initialize_artifacts()
database.initialize_database()
uri, name = mlflow_logger.resolve_mlflow_settings(configs)
mlflow_logger.setup_mlflow(uri, name)
print('DB        :', paths.DB_PATH)
print('MLflow URI:', uri)
print('Experiment:', name)
print('Tables    :', ['experiments', 'runs', 'evaluations', 'best_models'])

In [ ]:
import uuid
from src.evaluation.evaluator import EvaluationResult
exp_id = 'dryrun-' + uuid.uuid4().hex[:8]
database.insert_experiment(exp_id, 'DRYRUN', 'random_search', 1, 1, {}, {})
run_id = 'dryrun-' + uuid.uuid4().hex[:8]
database.create_run(run_id, exp_id, 'random_search', 11)
database.insert_evaluation(uuid.uuid4().hex, run_id, 'random_search', 11, 1,
                           '{"learning_rate": 0.001, "num_layers": 2}',
                           EvaluationResult(accuracy=0.9, macro_f1=0.5))
import pandas as pd
pd.DataFrame(database.fetch_all('evaluations')).tail(3)

In [ ]:
from src.models.search_space import SearchSpace
from src.models.architecture_builder import build_model
space = SearchSpace(configs['search_space'])
arch = space.sample_architecture()
model = build_model(arch, input_dim=187, num_classes=5)
x = torch.randn(8, 187)
print('sampled arch:', arch)
print('output shape:', tuple(model(x).shape), '(expected (8, 5))')